# CANADA Analysis

# Packages

In [45]:
#import gzip
import numpy as np
import os
import pandas as pd
import seaborn as sns

# 17th October 2024

Calculating the number of genes in the bottom 20% expression percentile of the L1000 [COLO-680N](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM886945) dataset

File downloaded to `DDesktop > Work > CANADA > Data`

Move file over to active `_code > canada` folder temporarily while working on it. Remove before syncing to GitHub.

## Files

In [75]:
# Import data
data = pd.read_csv('data.csv')
data.head()

,ID_REF,VALUE
0,1_at,5.9845
1,10_at,4.5031
2,100_at,8.2780
3,1000_at,4.4230
4,10000_at,5.0795


## Analysis

### Microarray Data

In [76]:
# Get data values
data_values = data['VALUE'].tolist()
# Calculate bottom 20th percentile threshold
pt = np.percentile(data_values, 20)
# Get data <= threshold
data_slice = data[data['VALUE'] <= pt]
# REPORT
# Statistics
len_data_slice = len(data_slice)
len_data = len(data)
percent_slice = len_data_slice / len_data * 100
# Print
print(f'Threhold for 20% percentile of expression calculated as: {pt}')
print(f'{percent_slice:.2f}% of records are <= this threshold ({len_data_slice:,}/{len_data:,})')
# Show data
data_slice.head()

Threhold for 20% percentile of expression calculated as: 3.9793
20.00% of records are <= this threshold (3,786/18,926)


,ID_REF,VALUE
7,10002_at,3.8912
8,10003_at,3.6664
10,100048912_at,3.8990
26,100126784_at,3.6305
27,100126791_at,3.8804


### Gene IDs

Sourced from GEO Accession Viewer [GPL15308](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL15308)

In [94]:
# Import gene IDs
ids = pd.read_csv('gene_ids.csv')
# Show data
ids.head()

,ID ORF Description
0,1_at 1 alpha-1-B glycoprotein
1,10_at 10 N-acetyltransferase 2 (arylami...
2,100_at 100 adenosine deaminase
3,"1000_at 1000 cadherin 2, type 1, N-cadherin..."
4,10000_at 10000 v-akt murine thymoma v...


In [95]:
# Rename column
ids.rename(columns = {ids.columns[0] : 'Column'}, inplace = True)
# Split cell data on first 2 instances of spaces
ids[['ID', 'ORF', 'Description']] = ids['Column'].str.split(' ', n = 2, expand = True)
# Drop 'Column'
ids.drop('Column', axis = 'columns', inplace = True)
# Rename ID > ID_REF for merging
ids.rename(columns = {'ID' : 'ID_REF'}, inplace = True)
# Show data
ids.head()

,ID_REF,ORF,Description
0,1_at,,1 alpha-1-B glycoprotein
1,10_at,,10 N-acetyltransferase 2 (arylamine N-ac...
2,100_at,,100 adenosine deaminase
3,1000_at,1000,"cadherin 2, type 1, N-cadherin (neuronal)"
4,10000_at,,10000 v-akt murine thymoma viral oncog...


In [106]:
# Combine with data_slice
percentile_data = pd.merge(data_slice, ids[['ID_REF', 'Description']], how = 'left', on = 'ID_REF')
# Strip values in 'Description' column
percentile_data['Description'] = percentile_data['Description'].str.strip()
# Split Description on first space
percentile_data[['desc_id', 'desc']] = percentile_data['Description'].str.split(' ', 1, expand = True)
# Drop columns
percentile_data.drop(['Description', 'desc_id'], axis = 'columns', inplace = True)
# Rename columns
percentile_data.rename(columns = {'desc' : 'Description'}, inplace = True)
# Strip 'Descripton' column values
percentile_data['Description'] = percentile_data['Description'].str.strip()
# Save data
percentile_data.to_csv('percentile_data.csv', index = False)
# Show data
percentile_data.head()

,ID_REF,VALUE,Description
0,10002_at,3.8912,"nuclear receptor subfamily 2, group E, member 3"
1,10003_at,3.6664,N-acetylated alpha-linked acidic dipeptidase 2
2,100048912_at,3.8990,CDKN2B antisense RNA (non-protein coding)
3,100126784_at,3.6305,uncharacterized LOC100126784
4,100126791_at,3.8804,eosinophil granule ontogeny transcript (non-pr...


# Next Analysis